# Chain of Thought (CoT) Prompting — AWS Bedrock Edition

## What is this notebook about?

This notebook teaches you **Chain of Thought (CoT) prompting** — a technique where instead of just asking the AI for an answer, you ask it to **show its reasoning step by step**.

Think of it like this:
- ❌ **Normal prompt:** "What is 15 × 4?" → AI says "60"
- ✅ **CoT prompt:** "Solve step by step: What is 15 × 4?" → AI says "15 × 2 = 30, double that = 60"

Same model. Same question. Just a different way of asking — and you get much better results on hard problems.

---

## What we will cover:
1. Setup — connect to Bedrock
2. Basic CoT — standard vs step-by-step
3. Advanced CoT — structured reasoning template
4. Comparative Analysis — see CoT fix a hard math problem
5. Logical Reasoning — apply CoT to a puzzle

---
## Step 1: Setup

We connect to **AWS Bedrock** using `boto3` and write a helper function `get_completion()` that handles both **Nova** and **Claude** models.

You only need to run this once — all other cells will use `get_completion()`.

In [ ]:
import boto3
import json

# Pick your model
# Option A: Amazon Nova (faster, cheaper)
MODEL_NAME = "us.amazon.nova-lite-v1:0"

# Option B: Claude Haiku (uncomment to switch)
# MODEL_NAME = "anthropic.claude-3-haiku-20240307-v1:0"

AWS_REGION = "us-east-1"

bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def get_completion(prompt, system='', model_name=MODEL_NAME):
    """Send a prompt to Bedrock and return the text response."""

    # Claude models
    if model_name.startswith("anthropic.claude"):
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        })
        response = bedrock.invoke_model(body=body, modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("content")[0].get("text")

    # Amazon Nova models
    elif "amazon.nova" in model_name:
        body_dict = {
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {
                "max_new_tokens": 2000,
                "temperature": 0.0,
                "top_p": 1,
            }
        }
        if system:
            body_dict["system"] = [{"text": system}]

        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("output").get("message").get("content")[0].get("text")

    else:
        raise ValueError(f"Unsupported model: {model_name}")


# Quick test
print(get_completion("Hello! Which model are you?"))

---
## Step 2: Basic CoT — Standard vs Step-by-Step

We ask the **exact same question** two ways:
- **Standard prompt** → just asks for the answer
- **CoT prompt** → adds 'step by step' to the ask

Watch how the output changes with just that one phrase added.

In [ ]:
question = "If a train travels 120 km in 2 hours, what is its average speed in km/h?"

# Standard: just ask for the answer
standard_prompt = f"Answer the following question concisely: {question}"

# CoT: ask to think step by step
cot_prompt = f"Answer the following question step by step concisely: {question}"

print("=" * 50)
print("STANDARD RESPONSE (no reasoning):")
print("=" * 50)
print(get_completion(standard_prompt))

print()
print("=" * 50)
print("CHAIN OF THOUGHT RESPONSE (with reasoning):")
print("=" * 50)
print(get_completion(cot_prompt))

---
## Step 3: Advanced CoT — Structured Reasoning Template

Instead of just saying 'step by step', we give the model a **specific structure** to follow for each step:
1. What are you calculating?
2. What formula will you use?
3. Do the calculation
4. Explain the result

More structure = more accurate reasoning on complex problems.

In [ ]:
def advanced_cot_prompt(question):
    """Wraps a question in a structured CoT template."""
    return f"""Solve the following problem step by step. For each step:
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {question}

Solution:"""


# A harder question that needs multi-step reasoning
complex_question = (
    "A car travels 150 km at 60 km/h, then another 100 km at 50 km/h. "
    "What is the average speed for the entire journey?"
)

print("ADVANCED COT RESPONSE:")
print("=" * 50)
print(get_completion(advanced_cot_prompt(complex_question)))

---
## Step 4: Comparative Analysis — Where CoT Really Shines

Now we throw a **much harder problem** at both approaches.

This water tank problem needs:
- Volume calculation (π × r² × h)
- Unit conversion (cubic meters → liters)
- Time calculation
- Final unit conversion (minutes → hours + minutes)

Watch how the **standard prompt guesses** but **CoT gets it right**.

In [ ]:
challenging_question = (
    "A cylindrical water tank with a radius of 1.5 meters and a height of 4 meters is 2/3 full. "
    "If water is being added at a rate of 10 liters per minute, how long will it take for the tank to overflow? "
    "Give your answer in hours and minutes, rounded to the nearest minute. "
    "(Use 3.14159 for pi and 1000 liters = 1 cubic meter)"
)

print("=" * 50)
print("STANDARD RESPONSE (just answer):")
print("=" * 50)
print(get_completion(f"Answer concisely: {challenging_question}"))

print()
print("=" * 50)
print("CHAIN OF THOUGHT RESPONSE (structured reasoning):")
print("=" * 50)
print(get_completion(advanced_cot_prompt(challenging_question)))

---
## Step 5: Logical Reasoning — Applying CoT to a Puzzle

CoT is not just for math. It works great for **logic puzzles** too.

Here we have 3 people — Amy, Bob, Charlie. One always lies, one always tells the truth, one alternates.
We need to figure out who is who from their statements.

We give the model an 8-step reasoning structure:
- List the facts
- Identify possible roles
- Test each combination
- Eliminate contradictions
- Give the final answer

Without CoT, the model often just guesses. With CoT, it works through it systematically.

In [ ]:
def logical_reasoning_prompt(scenario):
    """Wraps a logic puzzle in an 8-step analysis template."""
    return f"""Analyze the following logical puzzle. Follow these steps:

1. List the Facts - summarize all given info and characters
2. Identify Possible Roles - truth-teller, liar, alternator
3. Note the Constraints - rules and relationships
4. Generate Possible Scenarios - all combinations of roles
5. Test Each Scenario - check each statement for consistency
6. Eliminate Inconsistent Scenarios - discard contradictions
7. Conclude the Solution - which scenario survives
8. Provide a Clear Answer - state each person's role and why

Scenario:
{scenario}

Analysis:"""


logical_puzzle = (
    "In a room, there are three people: Amy, Bob, and Charlie. "
    "One of them always tells the truth, one always lies, and one alternates between truth and lies. "
    "Amy says: Bob is a liar. "
    "Bob says: Charlie alternates between truth and lies. "
    "Charlie says: Amy and I are both liars. "
    "Determine the nature (truth-teller, liar, or alternator) of each person."
)

print("LOGICAL REASONING - CoT RESPONSE:")
print("=" * 50)
print(get_completion(logical_reasoning_prompt(logical_puzzle)))

---
## Summary

| Technique | What changes | When to use |
|---|---|---|
| Standard prompt | Just asks for answer | Simple factual questions |
| Basic CoT | Adds 'step by step' | Medium complexity problems |
| Advanced CoT | Gives a structured template | Hard math / multi-step problems |
| Logical CoT | 8-step analysis framework | Logic puzzles / reasoning tasks |

### Key takeaway
You do not change the model. You do not change the code. You just **change how you phrase the prompt** — and the quality of reasoning improves dramatically.

This works the same way whether you are using Nova Lite, Claude Haiku, or any other Bedrock model.